In [15]:
def main(datasources, start_date, end_date):
    """
    因子构建主函数（演示：简单时序因子 —— N 日价格反转）

    评测时平台会自动替换 datasources / start_date / end_date 三个入参并调用本函数

    参数:
        datasources (dict): 数据源表名映射 {逻辑名: 物理表名}
                                "bar1m"     -> 分钟 K 线表
                                "financial" -> 财务数据表
        start_date (str): 开始时间
        end_date (str):   结束时间

    返回:
        pd.DataFrame: 因子数据，须包含三列 ['date', 'instrument', 'factor']，且不含 inf
    """
    import pandas as pd
    import dai

    # 从映射里取出本阶段实际的物理表名（切勿在 SQL 里硬编码表名，否则公榜/私榜无法切换）
    bar1m = datasources["bar1m"]

    # 时序因子需要向前多取若干天数据作为缓冲，才能算出区间开头几天的 N 日反转
    N = 5  # 反转窗口（交易日）

    # ===== 编写因子 SQL =====
    # 简单时序因子：N 日价格反转 = -(close_t / close_{t-N} - 1)
    # 过去一段涨得多的股票，短期倾向于回落，因此对反转收益取负号作为因子方向。
    # DAI 函数文档：https://bigquant.com/wiki/doc/Rceb2JQBdS
    sql = f"""
    WITH cte_daily AS (
        -- 分钟 K 线聚合成日频收盘价（取每个交易日最后一分钟的 close）
        SELECT
            instrument,
            strftime(date, '%Y-%m-%d') AS trading_day,
            last(close ORDER BY date) AS close
        FROM {bar1m}
        WHERE close > 0
        GROUP BY instrument, strftime(date, '%Y-%m-%d')
    ),
    cte_lag AS (
        SELECT
            *,
            -- 时序算子：按标的取 N 个交易日前的收盘价
            lag(close, {N}) OVER (PARTITION BY instrument ORDER BY trading_day) AS prev_close
        FROM cte_daily
    )
    SELECT
        CAST(trading_day AS DATETIME) AS date,
        instrument,
        -- 反转因子：对 N 日收益取负号
        -1 * (close / prev_close - 1) AS factor
    FROM cte_lag
    """

    # ===== 调用dai计算因子 =====
    # compression=True 会把 instrument 列转为 category 类型，显著降低内存占用
    df = dai.query(
        sql,
        filters={'date': [start_date, end_date]},
        compression=True,
    ).df()

    # ===== 对齐股票池 =====
    # 数据源保留了 2019 年至今所有成分股的数据以便计算时序因子，
    # 因此需与中证 1000 成分股做内连接，只保留当日属于成分股的标的
    # bigalpha_2026_instruments 已经收录了2019年以来的所有数据，不用替换
    stk_pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={'date': [start_date, end_date]},
    ).df()
    df = pd.merge(df, stk_pool, how='inner', on=['date', 'instrument'])

    return df


if __name__ == '__main__':
    from bigmodule import M
    import dai
    import structlog

    logger = structlog.get_logger()

    # 本地自测时自行构造数据源映射（评测时由平台注入，逻辑名固定为 "bar1m"/"financial"）
    datasources = {'bar1m': 'bigalpha_2026_stock_bar1m_selftest'}
    start_date = '2024-01-01 00:00:00'
    end_date = '2024-10-31 23:59:59'

    # 计算因子
    logger.info(f"计算因子，区间：{start_date} ~ {end_date}")
    factor_data = main(datasources, start_date, end_date)

    # 读取平台因子库用于回归评估，您可以换成自己的因子库
    logger.info(f"读取因子库，区间：{start_date} ~ {end_date}")
    factor_pool = dai.query(
        "SELECT * FROM bigalpha_2026_factorlib",
        filters={'date': [start_date, end_date]},
    ).df()

    # 评估系统：
    # process_pools=False 表示不对因子库再做预处理（bigalpha_2026_factorlib 已处理过）
    # show=True 表示画出评估图表
    result = M.bigalpha_eval._latest(
        factor_data=factor_data,
        factor_pool=factor_pool,
        start_date=start_date,
        end_date=end_date,
        process_pools=False,
        show=True,
    )

[2026-07-16 16:31:10] [info     ] 计算因子，区间：2024-01-01 00:00:00 ~ 2024-10-31 23:59:59
[2026-07-16 16:31:12] [info     ] 读取因子库，区间：2024-01-01 00:00:00 ~ 2024-10-31 23:59:59
[2026-07-16 16:31:13] [warning  ] bigalpha_eval._latest version='v4' (use ._latest for dev only, not for prod)
[2026-07-16 16:31:14] [info     ] bigalpha_eval.v4 开始运行 ..
[2026-07-16 16:31:15] [info     ] 对齐中证1000历史成分后，官方评估窗口: 2024-01-02 至 2024-10-31
[2026-07-16 16:31:15] [info     ] ========== 数据检查 ==========
[2026-07-16 16:31:15] [info     ] 通过：列名检查（date/instrument + 至少 1 个因子列） factor_cols=['factor', 'close', 'volume', 'amount', 'turn', 'change_ratio', 'daily_return', 'momentum_5', 'reversal_5', 'volatility_5', 'total_market_cap', 'float_market_cap', 'pe_ttm', 'pb', 'ps_ttm', 'sma_20', 'ema_20', 'macd_diff_12_26_9', 'macd_dea_12_26_9', 'macd_hist_12_26_9', 'rsi_12', 'kdj_k_9_3_3', 'kdj_d_9_3_3', 'bias_20', 'cci_14', 'atr_14', 'roe_avg_ttm', 'roa_avg_ttm', 'gross_profit_rate_ttm', 'net_profit_rate_ttm', 'debt_to_asset_l

DataValidationError: 覆盖度检查失败：因子 factor 单日缺失率 > 40%；详情={'factor': 'factor', 'sample': {'2024-01-02': 1.0, '2024-01-03': 1.0, '2024-01-04': 1.0, '2024-01-05': 1.0, '2024-01-08': 1.0}, 'total_days': 5}

In [16]:
factor_data[factor_data['date']=='2024-01-02']

,date,instrument,factor
0,2024-01-02,002911.SZ,NaN
1,2024-01-03,002911.SZ,NaN
2,2024-01-04,002911.SZ,NaN
3,2024-01-05,002911.SZ,NaN
4,2024-01-08,002911.SZ,NaN
...,...,...,...
198816,2024-10-25,600366.SH,-0.058915
198817,2024-10-28,600366.SH,-0.069444
198818,2024-10-29,600366.SH,-0.016591
198819,2024-10-30,600366.SH,-0.016492
